<a href="https://colab.research.google.com/github/LizLian/from_scratch_2025/blob/main/gpt_dev_20260312.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2025-12-11 15:42:28--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com... failed: Name or service not known.
wget: unable to resolve host address 'raw.githubusercontent.com'


In [ ]:
from google3.pyglib import gfile

In [ ]:
# read it in to inspect it
with gfile.Open('/cns/wf-d/home/lizliang/dataset/from_scratch/input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [ ]:
print("length of dataset in characters: ", len(text))

length of dataset in characters:  1115394


In [ ]:
# let's look at the first 1000 characters
print(text[:100])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [ ]:
# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [ ]:
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s] # encoder: takes a strign, outputs a list of integers
decode = lambda l: "".join([itos[i] for i in l]) # decoder: takes a list of integers, outputs a string

print(encode("hii there"))
print(decode([46, 47, 47, 1, 58, 46, 43, 56, 43]))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [ ]:
# encode the entire text dataset and store it into a torch.Tensor
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:100])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [ ]:
# split up the data into train and validation sets
n = int(len(data) * 0.9)
train_data = data[:n]
val_data = data[n:]

In [ ]:
def get_batch(split):
  # generate a small batch of data of inputs x and targets y
  data = train_data if split == "train" else val_data
  ix = torch.randint(len(data) - block_size, (batch_size,))
  x = torch.stack([data[i:i+block_size] for i in ix])
  y = torch.stack([data[i+1:i+block_size+1] for i in ix])
  x, y = x.to(device), y.to(device)
  return x, y

@torch.no_grad()
def estimate_loss():
  out = {}
  model.eval()
  for split in ["train", "val"]:
    losses = torch.zeros(eval_iters)
    for k in range(eval_iters):
      X, Y = get_batch(split)
      logits, loss = model(X, Y)
      losses[k] = loss.item()
    out[split] = losses.mean()
  model.train()
  return out

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

batch_size = 16 # 64
block_size = 32 #256
max_iters = 5000
eval_interval = 100
learning_rate = 1e-3
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
eval_iters = 200
n_embd = 64 # 384
n_head = 4
n_layer = 4
dropout = 0.2
vocab_size = len(chars)

cuda


In [ ]:
# @title Attention Implementation
# self-attention: keys, queries and values are from the same source (e.g. a decoder only model).
# cross-attention: keys and values are from different source (encoder), values are from decoder.

# scale attention softmax(Q@K^t/sqrt(headsize))@V
# why? 1) if Q and K are very large, the softmax result will be at its tail (peaky&one-hot) 2) keep weight variance to 1, ensuring healthy gradients

class Head(nn.Module):

  def __init__(self, head_size):
    super().__init__()
    self.key = nn.Linear(n_embd, head_size, bias=False)
    self.value = nn.Linear(n_embd, head_size, bias=False)
    self.query = nn.Linear(n_embd, head_size, bias=False)
    self.register_buffer("tril", torch.tril(torch.ones((block_size, block_size))))

  def forward(self, x):
    B, T, C = x.shape
    # print(T, C)
    q = self.query(x) # (B, T, head_size)
    k = self.key(x)
    v = self.value(x) # (B, T, head_size)
    weight = q @ k.transpose(1,2) / C**0.5 # BxTxT
    weight = weight.masked_fill(self.tril[:T, :T]==0, float("-inf"))
    weight = F.softmax(weight, dim=-1)
    out = weight @ v
    return out


class MultiHeadAttention(nn.Module):
  """ multiple heads of self-attention in parallel """

  def __init__(self, num_heads, head_size):
    super().__init__()
    self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
    self.proj = nn.Linear(n_embd, n_embd)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    out = torch.cat([h(x) for h in self.heads], dim=-1)
    out = self.dropout(self.proj(out))
    return out

In [ ]:
# @title Implement FeedForward Network

class FeedForward(nn.Module):
  """linear layer followed by relu."""

  def __init__(self, n_embd):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(n_embd, 4 * n_embd),
        nn.ReLU(),
        nn.Linear(4 * n_embd, n_embd),
        nn.Dropout(dropout)
    )

  def forward(self, x):
    out = self.net(x)
    return out


class Block(nn.Module):
  def __init__(self, n_embd, n_head):
    super().__init__()
    head_size = n_embd // n_head
    self.sa = MultiHeadAttention(n_head, head_size)
    self.ln1 = nn.LayerNorm(n_embd)
    self.ln2 = nn.LayerNorm(n_embd)
    self.ffwd = FeedForward(n_embd)

  def forward(self, x):
    x = x + self.sa(self.ln1(x))
    x = x + self.ffwd(self.ln2(x))
    return x


class LayerNorm1D(nn.Module):

  def __init__(self, dim, eps=1e-5):
    # self.gamma =
    # self.beta =
    pass

  def __call__(self, x):
    pass

  def parameters(self):
    return []


In [ ]:
# @title Bigram language model pytorch implementation

class BigramLanguageModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
    self.position_embedding_table = nn.Embedding(block_size, n_embd)
    self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
    self.ln_f = nn.LayerNorm(n_embd)
    self.lm_head = nn.Linear(n_embd, vocab_size)

  def forward(self, idx, targets=None):
    B, T = idx.shape

    tok_emb = self.token_embedding_table(idx)
    pos_emb = self.position_embedding_table(torch.arange(T, device=device))
    x = tok_emb + pos_emb # (B, T, C)
    x = self.blocks(x)
    x = self.ln_f(x)
    logits = self.lm_head(x) # (B, T, vocab_size)
    loss = None
    if targets is not None:
      B, T, C = logits.shape
      logits = logits.view(B*T, C)
      targets = targets.view(-1)
      loss = F.cross_entropy(logits, targets)

    return logits, loss

  def generate(self, idx, max_new_tokens):
    # idx is (B, T) array of indices in the current context
    for _ in range(max_new_tokens):
      # crop idx to the last block_size tokens
      idx_cond = idx[:, -block_size:]
      logits, loss = self(idx_cond)
      # focus only on the last time step
      logits = logits[:, -1, :] # becomes (B, C)
      probs = F.softmax(logits, dim=-1) # (B, C)
      # sample from the distribution
      idx_next = torch.multinomial(probs, num_samples=1)
      idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
    return idx

model = BigramLanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, "M parameters")


0.209729 M parameters


In [ ]:
# @title Training Loop
optimizer = torch.optim.AdamW(m.parameters(), lr=learning_rate)
for iter in range(max_iters):
  # every once in a while evaluate the loss on train and eval sets
  if iter % eval_interval == 0 or iter == max_iters - 1:
    losses = estimate_loss()
    print(f"step {iter}: train loss {losses["train"]:.4f}, val loss {losses["val"]:.4f}")

  # sample a batch of data
  xb, yb = get_batch("train")

  # evaluate the loss
  logits, loss = m(xb, yb)
  optimizer.zero_grad(set_to_none=True)
  loss.backward()
  optimizer.step()

step 0: train loss 4.3505, val loss 4.3428
step 100: train loss 2.6614, val loss 2.6890
step 200: train loss 2.5159, val loss 2.5246
step 300: train loss 2.4458, val loss 2.4679
step 400: train loss 2.3910, val loss 2.3996
step 500: train loss 2.3432, val loss 2.3523
step 600: train loss 2.3026, val loss 2.3027
step 700: train loss 2.2612, val loss 2.2701
step 800: train loss 2.2221, val loss 2.2509
step 900: train loss 2.1958, val loss 2.2211
step 1000: train loss 2.1747, val loss 2.1977
step 1100: train loss 2.1445, val loss 2.1572
step 1200: train loss 2.1214, val loss 2.1507
step 1300: train loss 2.1087, val loss 2.1426
step 1400: train loss 2.0749, val loss 2.1074
step 1500: train loss 2.0485, val loss 2.0924
step 1600: train loss 2.0331, val loss 2.0819
step 1700: train loss 2.0158, val loss 2.0806
step 1800: train loss 2.0054, val loss 2.0627
step 1900: train loss 1.9822, val loss 2.0562
step 2000: train loss 1.9798, val loss 2.0437
step 2100: train loss 1.9604, val loss 2.0463


In [ ]:
# @title Generate from the model
context = torch.zeros((1,1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))


Thoughter, should uspay:
I will mathery so his kineme; I'll at of in a would think dou't a rove, dryalt vens!

MORTINIUS: leselfs, unsure? Shal good-mer Pince Vommed, of then:
Whone you' are resents me: pictleder's cattion-my sit'!

Nor min not so that; I wisNord bundstled' fare his!

QUEEN EDWABPUKENTHER:
So surre theine thou sed to Lo'ly werwer.

Llower?
He tere menself YorSwer
Thou we sult 'ear'vir herainful, I sauly.

GLOUCH:
What thomy the warl?
And, with  say!

BEYV:
Us mire may coldom of 
